[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/deep-learning-diagnostics-and-improvement/blob/main/practice/11_optimizer_updates.ipynb)

# 11. Optimizer updates — paper-faithful core updates

같은 작은 parameter와 gradient에 optimizer update를 직접 적용해 **state가 어떻게 누적되고, 실제 ΔW가 어떻게 만들어지는지** 비교한다.

특히 이전 버전에서 빠져 있던 두 부분을 복구했다.

- Prodigy: 단순히 `D`를 곱하는 것이 아니라 **D 추정량 자체를 step마다 갱신**한다.
- Muon: 단순 Newton–Schulz 한 번이 아니라 **momentum → Nesterov update → quintic Newton–Schulz → aspect-ratio scaling → parameter update** 전체 사슬을 본다.


In [ ]:
import math

import torch

torch.manual_seed(7)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("device:", device)
print("torch:", torch.__version__)


## 1. SGD and momentum

SGD는 현재 gradient만 사용하고, momentum은 이전 gradient 정보를 state에 누적한다.


In [ ]:
weight = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    device=device,
)
gradient = torch.tensor(
    [[0.2, -0.4], [1.0, 0.5]],
    device=device,
)
learning_rate = 0.1

sgd_weight = weight - learning_rate * gradient
print("SGD weight:\n", sgd_weight)

momentum = torch.zeros_like(weight)
beta = 0.9
momentum_weight = weight.clone()

for step in range(3):
    momentum = beta * momentum + gradient
    momentum_weight = momentum_weight - learning_rate * momentum

    print("step:", step)
    print("momentum:\n", momentum)
    print("weight:\n", momentum_weight)


## 2. Adam and AdamW

Adam은 first/second moment EMA와 bias correction을 사용한다. AdamW는 weight decay를 gradient 안에 섞지 않고 parameter에 직접 decoupled 형태로 적용한다.


In [ ]:
weight = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    device=device,
)
first_moment = torch.zeros_like(weight)
second_moment = torch.zeros_like(weight)

beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

for step in range(1, 4):
    first_moment = (
        beta1 * first_moment
        + (1 - beta1) * gradient
    )
    second_moment = (
        beta2 * second_moment
        + (1 - beta2) * gradient.square()
    )

    first_unbiased = first_moment / (1 - beta1 ** step)
    second_unbiased = second_moment / (1 - beta2 ** step)

    update = first_unbiased / (second_unbiased.sqrt() + epsilon)

    adam_weight = weight - learning_rate * update
    adamw_weight = (
        weight * (1 - learning_rate * 0.01)
        - learning_rate * update
    )

    print("step:", step)
    print("Adam ΔW:\n", adam_weight - weight)
    print("AdamW ΔW:\n", adamw_weight - weight)


## 3. Lion

Lion의 중요한 차이는 update magnitude를 adaptive denominator로 만들지 않고 **momentum과 gradient 조합의 sign**을 parameter update 방향으로 사용한다는 점이다.


In [ ]:
momentum = torch.zeros_like(gradient)
beta1 = 0.9
beta2 = 0.99

update_direction = (
    beta1 * momentum
    + (1 - beta1) * gradient
).sign()

momentum = (
    beta2 * momentum
    + (1 - beta2) * gradient
)

print("sign update:\n", update_direction)
print("new momentum:\n", momentum)


## 4. Prodigy: D-adaptation state is part of the optimizer

이전 버전처럼 임의의 `D=0.02`를 그냥 곱하면 Prodigy가 아니다. 공식 구현의 핵심은 `p0`, 누적 state `s`, numerator/denominator를 이용해 `d_hat`을 만들고, 그 값으로 다음 adapted step size `d`를 갱신하는 것이다.

아래 코드는 distributed/FSDP, slicing, weight decay 같은 주변 기능은 빼고 **D 추정 → Adam-style moments → parameter update**의 핵심 state transition을 작은 tensor에서 유지한다.


In [ ]:
class TinyProdigyState:
    def __init__(self, parameter):
        self.p0 = parameter.detach().clone()
        self.s = torch.zeros_like(parameter)
        self.exp_avg = torch.zeros_like(parameter)
        self.exp_avg_sq = torch.zeros_like(parameter)

        self.d = 1e-3
        self.d_max = self.d
        self.d_numerator = 0.0
        self.step = 0


def tiny_prodigy_step(
    parameter,
    gradient,
    state,
    base_lr=1.0,
    beta1=0.9,
    beta2=0.999,
    beta3=None,
    d_coef=1.0,
    epsilon=1e-8,
):
    if beta3 is None:
        beta3 = math.sqrt(beta2)

    d = state.d
    d0 = 1e-3
    adapted_lr = d * base_lr

    displacement = state.p0 - parameter
    delta_numerator = (
        (d / d0)
        * adapted_lr
        * torch.sum(gradient * displacement).item()
    )

    state.d_numerator = (
        beta3 * state.d_numerator
        + delta_numerator
    )

    state.s = (
        beta3 * state.s
        + ((d / d0) * adapted_lr) * gradient
    )

    d_denom = state.s.abs().sum().item()

    if d_denom > 0:
        d_hat = d_coef * state.d_numerator / d_denom
        state.d_max = max(state.d_max, d_hat)
        state.d = max(state.d, state.d_max)

    state.exp_avg = (
        beta1 * state.exp_avg
        + state.d * (1 - beta1) * gradient
    )
    state.exp_avg_sq = (
        beta2 * state.exp_avg_sq
        + state.d**2 * (1 - beta2) * gradient.square()
    )

    denominator = state.exp_avg_sq.sqrt() + state.d * epsilon
    update = state.exp_avg / denominator

    parameter = parameter - base_lr * update
    state.step += 1

    return parameter, state


parameter = torch.tensor(
    [[1.0, -1.0], [0.5, 2.0]],
    device=device,
)
state = TinyProdigyState(parameter)

gradient_sequence = [
    gradient,
    0.7 * gradient,
    0.4 * gradient,
]

for step_gradient in gradient_sequence:
    parameter, state = tiny_prodigy_step(
        parameter,
        step_gradient,
        state,
    )

    print("step:", state.step)
    print("estimated d:", state.d)
    print("parameter:\n", parameter)


## 5. Muon: momentum → orthogonalization → update

Muon은 hidden matrix weight에 대해 먼저 momentum/Nesterov update를 만들고, 그 2D update를 Newton–Schulz로 zeroth-power/orthogonalized 방향에 가깝게 바꾼다. 공식 구현은 단순 cubic iteration이 아니라 `(a,b,c)=(3.4445,-4.7750,2.0315)`인 **quintic Newton–Schulz**를 사용한다.


In [ ]:
def zeropower_via_newton_schulz5(matrix, steps=5):
    a = 3.4445
    b = -4.7750
    c = 2.0315

    x = matrix.to(torch.bfloat16)
    transposed = x.size(-2) > x.size(-1)

    if transposed:
        x = x.mT

    x = x / (
        x.norm(dim=(-2, -1), keepdim=True)
        + 1e-7
    )

    for _ in range(steps):
        gram = x @ x.mT
        polynomial = b * gram + c * (gram @ gram)
        x = a * x + polynomial @ x

    if transposed:
        x = x.mT

    return x.to(matrix.dtype)


def tiny_muon_update(
    gradient,
    momentum_buffer,
    beta=0.95,
    ns_steps=5,
):
    momentum_buffer.lerp_(
        gradient,
        1 - beta,
    )

    nesterov_update = gradient.lerp(
        momentum_buffer,
        beta,
    )

    orthogonalized = zeropower_via_newton_schulz5(
        nesterov_update,
        steps=ns_steps,
    )

    rows = orthogonalized.size(-2)
    cols = orthogonalized.size(-1)
    aspect_scale = max(1.0, rows / cols) ** 0.5

    return aspect_scale * orthogonalized


weight = torch.tensor(
    [[1.0, 2.0], [3.0, 4.0]],
    device=device,
)
gradient = torch.tensor(
    [[0.2, -0.5], [0.7, 0.1]],
    device=device,
)
momentum_buffer = torch.zeros_like(weight)

muon_direction = tiny_muon_update(
    gradient,
    momentum_buffer,
)

muon_lr = 0.02
weight_decay = 0.01

new_weight = weight * (1 - muon_lr * weight_decay)
new_weight = new_weight - muon_lr * muon_direction

print("momentum buffer:\n", momentum_buffer)
print("orthogonalized update:\n", muon_direction)
print("singular values:", torch.linalg.svdvals(muon_direction))
print("new weight:\n", new_weight)


## References and provenance

**Adam / AdamW** — Kingma & Ba; Loshchilov & Hutter. first/second moments, bias correction, decoupled weight decay를 반영했다.

**Lion** — Chen et al., *Symbolic Discovery of Optimization Algorithms*. sign-based update와 momentum state를 반영했다.

**Prodigy** — Mishchenko & Defazio, *Prodigy: An Expeditiously Adaptive Parameter-Free Learner*, ICML 2024 및 공식 `konstmish/prodigy` 구현. `p0`, `s`, numerator/denominator, `d_hat`, Adam-style moments의 연결을 반영했다.

**Muon** — Keller Jordan의 공식 `KellerJordan/Muon` 구현. current implementation의 momentum/Nesterov update, quintic Newton–Schulz coefficients, aspect-ratio scaling, AdamW-style decay를 반영했다. 실제 사용에서는 hidden matrix weights에 Muon을 쓰고 embedding/head/bias/gain에는 AdamW 계열을 쓰는 구성이 일반적이다.
